In [1]:
import subprocess
from pathlib import Path
import pandas as pd
import bioframe as bf
import numpy as np

######### Paths
anno_dir = Path("/abyss/dlafonta/data/K562_data/annotation")
anno_dir.mkdir(parents=True, exist_ok=True)
gtf_url  = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_50/gencode.v50.annotation.gtf.gz"
gtf_path = anno_dir / "gencode.v50.annotation.gtf.gz"
bed_path = anno_dir / "gencode.v50.transcripts.bed"   # cached parse

bed_cols = ["chrom", "start", "end", "strand", "gene_id", "ENST",
            "span_gtf", "gene_type", "transcript_type", "gene_name"]

######### Build cache once; reuse thereafter
if bed_path.exists():
    tx_data = pd.read_csv(bed_path, sep="\t")
else:
    if not gtf_path.exists():
        subprocess.run(["wget", "-O", str(gtf_path), gtf_url], check=True)

    gtf = bf.read_table(str(gtf_path), schema="gtf", comment="#")
    tx = gtf[gtf["feature"] == "transcript"].copy()

    def attr(series, key):
        return series.str.extract(rf'{key} "([^"]+)"')[0]

    tx["gene_id"]         = attr(tx["attributes"], "gene_id").str.split(".").str[0]
    tx["ENST"]            = attr(tx["attributes"], "transcript_id").str.split(".").str[0]
    tx["gene_type"]       = attr(tx["attributes"], "gene_type")
    tx["transcript_type"] = attr(tx["attributes"], "transcript_type")
    tx["gene_name"]       = attr(tx["attributes"], "gene_name")
    tx["span_gtf"]        = tx["end"] - tx["start"]

    tx_data = tx[bed_cols].copy()
    tx_data.to_csv(bed_path, sep="\t", index=False)   # header kept for self-documentation

##### Merge salmon 

sr = pd.read_csv("/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/salmon/K562_salmon/quant.sf", sep="\t")
sr = sr.rename(columns={"Name": "ENST", "NumReads": "counts"})

sr["ENST"] = sr["ENST"].str.split(".").str[0]


####### add ENSG via your transcript->gene table (tx2gene)
tx_data = tx_data.merge(sr, on="ENST")


# TPM_y is the Salmon per-transcript TPM; counts is Salmon NumReads.
# Rank within each gene by Salmon TPM (TPM_y), tie-broken by Length.

######### Identify which genes have any expression signal at all
gene_max_tpm = tx_data.groupby("gene_id")["TPM"].transform("max")
tx_data["_gene_expressed"] = gene_max_tpm > 0

def pick_dominant(df):
    if df["TPM"].max() > 0:
        # expressed gene: highest Salmon TPM (tie-break: longest, then lowest ENST for determinism)
        return df.sort_values(["TPM", "Length", "ENST"],
                              ascending=[False, False, True]).iloc[0]
    else:
        # dropout gene: longest transcript
        return df.sort_values(["Length", "ENST"],
                              ascending=[False, True]).iloc[0]

dominant = (tx_data.groupby("gene_id", group_keys=False)
                   .apply(pick_dominant)
                   .reset_index(drop=True))

# label provenance so the fallback is auditable
dominant["selection"] = np.where(dominant["TPM"] > 0, "expressed_maxTPM", "dropout_longest")



##Sanity checks

########  one row per ENSG (no duplicate genes in the output) 
dup = dominant["gene_id"].duplicated().sum()
print(f"Duplicate gene_id in dominant: {dup}")
assert dominant["gene_id"].is_unique, "dominant has duplicate genes!"

######## every ENSG from tx_data is retained ----
genes_in  = set(tx_data["gene_id"].unique())
genes_out = set(dominant["gene_id"].unique())

missing = genes_in - genes_out      # in input, lost in output
extra   = genes_out - genes_in      # in output, not in input (should be empty)


################ STRATIFY: SPIN group  ×  expression bin


spin = pd.read_csv(
    "/abyss/dlafonta/data/K562_data/SPIN/SPIN_50kb_rebinned.bed",
    sep="\t",
) 

# assign each transcript a SPIN state by overlap (e.g. TSS bin, or majority)
# TSS-based is cleanest and deterministic:
dominant["tss"] = np.where(dominant["strand"] == "+",
                           dominant["start"], dominant["end"] - 1)
tss = dominant.assign(start=dominant["tss"], end=dominant["tss"] + 1)

ov = bf.overlap(tss, spin, how="left",
                suffixes=("", "_spin"))[["gene_id", "SPIN_spin"]]
ov = ov.rename(columns={"SPIN_spin": "SPIN"}).drop_duplicates("gene_id")

dominant = dominant.merge(ov, on="gene_id", how="left")

d = dominant.copy()

# --- SPIN strata ---
spin_group = pd.Series(pd.NA, index=d.index, dtype="object")
spin_group[d["SPIN"] == "Speckle"] = "Speckle"
spin_group[d["SPIN"].isin(["Interior_Act2", "Interior_Act3"])] = "Interior_Act23"
d["spin_group"] = spin_group



# --- expression bins (intentional gap: 0 < TPM_x <= 5 excluded) ---
expr_bin = pd.Series(pd.NA, index=d.index, dtype="object")
expr_bin[d["TPM"] == 0] = "silent"
expr_bin[d["TPM"] > 5]  = "expressed"
d["expr_bin"] = expr_bin


# keep only rows in both a SPIN group and an expression bin
strat = d.dropna(subset=["spin_group", "expr_bin"]).copy()

# genomic span = end - start
strat = strat.copy()
strat["span"] = strat["end"] - strat["start"]

# filter: span > 10 kb
strat_10kb = strat[strat["span"] > 10_000].copy()




######### WRITE BED FILES  (no header; chr,start,end,ENSG,ENST,strand,Length)


out_dir = Path("/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions")
#out_dir.mkdir(parents=True, exist_ok=True)   # create it if needed

cols = ["chrom", "start", "end", "gene_id", "ENST", "strand", "Length"]

groups = {
    "Speckle_silent":         (strat["spin_group"] == "Speckle")        & (strat["expr_bin"] == "silent"),
    "Speckle_expressed":      (strat["spin_group"] == "Speckle")        & (strat["expr_bin"] == "expressed"),
    "InteriorAct23_silent":   (strat["spin_group"] == "Interior_Act23") & (strat["expr_bin"] == "silent"),
    "InteriorAct23_expressed":(strat["spin_group"] == "Interior_Act23") & (strat["expr_bin"] == "expressed"),
}

for name, mask in groups.items():
    out = strat.loc[mask, cols]
    path = out_dir / f"K562_dominant_{name}.bed"
    out.to_csv(path, sep="\t", header=False, index=False)
    print(f"{path}: {len(out)} rows")

total = sum(m.sum() for m in groups.values())


out_dir = Path("/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions")

cols = ["chrom", "start", "end", "gene_id", "ENST", "strand", "Length"]

groups_10kb = {
    "Speckle_silent":          (strat_10kb["spin_group"] == "Speckle")        & (strat_10kb["expr_bin"] == "silent"),
    "Speckle_expressed":       (strat_10kb["spin_group"] == "Speckle")        & (strat_10kb["expr_bin"] == "expressed"),
    "InteriorAct23_silent":    (strat_10kb["spin_group"] == "Interior_Act23") & (strat_10kb["expr_bin"] == "silent"),
    "InteriorAct23_expressed": (strat_10kb["spin_group"] == "Interior_Act23") & (strat_10kb["expr_bin"] == "expressed"),
}

for name, mask in groups_10kb.items():
    out = strat_10kb.loc[mask, cols]
    path = out_dir / f"K562_dominant_{name}_10kbfilter2.bed"
    out.to_csv(path, sep="\t", header=False, index=False)
    print(f"{path}: {len(out)} rows")

total = sum(m.sum() for m in groups_10kb.values())


Duplicate gene_id in dominant: 0
/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_Speckle_silent.bed: 4273 rows
/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_Speckle_expressed.bed: 1469 rows
/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_InteriorAct23_silent.bed: 10282 rows
/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_InteriorAct23_expressed.bed: 2100 rows
/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_Speckle_silent_10kbfilter2.bed: 733 rows
/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_Speckle_expressed_10kbfilter2.bed: 566 rows
/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_InteriorAct23_silent_10kbfilter2.bed: 2180 rows
/sharehome/dlafonta/manuscripts/RNA/Figure_zoom/bbi_stacks/regions/K562_dominant_InteriorAct23_expressed_10kbfilter2.bed: 1553 r

/tmp/ipykernel_1706812/3462678370.py:69: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dominant = (tx_data.groupby("gene_id", group_keys=False)
